# Validate Somersalo scaling law

## Load data

In [ ]:
from functools import partial

from multipac_testbench import AveragedThresholdSet, TestCampaign, ThresholdSet
from multipac_testbench.data import config_path
from multipac_testbench.data.multipactor_tests import tests_140
from multipac_testbench.instruments import CurrentProbe, ForwardPower
from multipac_testbench.util.multipactor_detectors import quantity_is_above_threshold

freqs = (140.0, 140.0, 140.0, 140.0)
swrs = (4.0, 3.0, 2.0, 1.0)

test_campaign =  TestCampaign.from_filepaths(
    tests_140,
    freqs,
    swrs,
    config_path,
    is_raw=True
)

## Calculate multipactor thresholds

In [ ]:
current_multipactor_criterions = {'threshold': 14., 'minimum_number_of_points': 1}
current_multipac_detector = partial(quantity_is_above_threshold, **current_multipactor_criterions)

As Somersalo scaling law concerns power thresholds, we look for "global" multipactor criterions:

In [ ]:
merged = test_campaign.determine_thresholds(
    current_multipac_detector,
    CurrentProbe,
    threshold_predicate=lambda t: t.sample_index > 300,  # Only consider Thresholds measured after 300th power step
    threshold_reducer="any",                   # Keyword to merge the multipactor zones
)

## Check Somersalo scaling law

### Naive thresholds

In [ ]:
_ = test_campaign.check_somersalo_scaling_law(merged, figsize=(8, 8))

### Extreme thresholds

You can filter out the thresholds, to keep only one lower and one upper threshold per half-power cycle.

In [ ]:
extreme = {test: ThresholdSet.extreme(threshold_set) for test, threshold_set in merged.items()}
_ = test_campaign.check_somersalo_scaling_law(extreme, figsize=(8, 8))

### Average thresholds

In [ ]:
averaged = {test: AveragedThresholdSet.from_threshold_set(threshold_set) for test, threshold_set in merged.items()}
_ = test_campaign.check_somersalo_scaling_law(averaged, figsize=(8, 8))